# Demonstration of Alternative Urban Development Scenarios

This notebook reproduces five representative alternatives from the aggregated Prompt 3 Pareto front, verifies their saved land-value gains on the same 333 blocks used during optimization, and displays publication-ready spatial figures inline.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import Markdown, display

from urbanomy.methods.land_value_modeling import (
    LandPriceEstimator, ScenarioTEPModifier, plot_scenario_impact,
)

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if not (ROOT / "examples").is_dir():
    raise FileNotFoundError("Run this notebook from the repository root or examples/.")

ROOT

## 1. Baseline data used by the optimization

The 95th-percentile filter below is identical to the preprocessing in `NSGA2.ipynb` and `NSGA2_LLM.ipynb`. The assertion prevents figures from being produced from the former 351-block dataset.

In [ ]:
feature_cols = [
    "residential", "business", "recreation", "industrial",
    "transport", "special", "agriculture", "land_use", "share",
    "footprint_area", "build_floor_area", "living_area",
    "non_living_area", "population", "site_area", "fsi", "gsi",
    "mxi", "l", "morphotype", "area_accessibility",
]
cat_features = ["land_use", "morphotype"]
numeric_features = [column for column in feature_cols if column not in cat_features]

baseline_blocks = gpd.read_file(ROOT / "examples/data/blocks_agg_with_indicators.geojson")
baseline_blocks["residential"] = baseline_blocks["residential"].astype("float64")
baseline_blocks = baseline_blocks.drop(columns=["site_area"], errors="ignore").copy()
metric_blocks = baseline_blocks.to_crs(baseline_blocks.estimate_utm_crs())
baseline_blocks["site_area"] = metric_blocks.geometry.area.values
baseline_blocks["id"] = baseline_blocks.index

model = CatBoostRegressor()
model.load_model(ROOT / "examples/data/catboost_land_value_no_services.cbm")
estimator_kwargs = {
    "model": model,
    "orig_features": numeric_features + cat_features,
    "categorical_features": cat_features,
    "use_service_features": False,
}

blocks_pred = LandPriceEstimator(blocks=baseline_blocks, **estimator_kwargs).predict()
blocks_pred["land_value_per_100m2"] = (
    blocks_pred["land_value"] / blocks_pred["site_area"] * 100
)
blocks_pred = blocks_pred.replace([np.inf, -np.inf], np.nan).fillna(0)
p95 = blocks_pred["land_value_per_100m2"].quantile(0.95)
blocks_clean = blocks_pred.loc[blocks_pred["land_value_per_100m2"] <= p95].copy()
blocks_clean["id"] = blocks_clean.index

assert len(blocks_clean) == 333, f"Expected 333 optimization blocks, got {len(blocks_clean)}"
assert blocks_clean["id"].is_unique
print(f"Baseline prepared: {len(blocks_clean)} blocks (from {len(baseline_blocks)} raw blocks).")

## 2. Reproducible selection of five Prompt 3 alternatives

The first three alternatives maximize one objective each. The balanced alternative minimizes Euclidean distance to the ideal point `(1, 1, 1)` after min-max normalization of all three maximized objectives. A fifth alternative maximizes the recalculated post-development value of target block 86.

In [ ]:
front_path = ROOT / "examples/best_pareto_fronts_10_seeds/prompt_3/prompt_3_best_pareto_front_10_seeds.jsonl"
front = pd.read_json(front_path, lines=True)
front["algorithm_variant"] = "NSGA-II with LLM"
no_llm_path = ROOT / "examples/best_pareto_fronts_10_seeds/prompt_3/no_llm_scored_by_prompt_3_best_10_seeds.jsonl"
no_llm_front = pd.read_json(no_llm_path, lines=True)
no_llm_front["algorithm_variant"] = "NSGA-II without LLM"
objective_cols = ["investor_npv", "land_value_gain", "llm score"]
normalized = (front[objective_cols] - front[objective_cols].min()) / (
    front[objective_cols].max() - front[objective_cols].min()
)
distance_to_ideal = np.sqrt(((1 - normalized) ** 2).sum(axis=1))

selection_rules = [
    ("Maximum NPV", front["investor_npv"].idxmax()),
    ("Maximum land-value gain", front["land_value_gain"].idxmax()),
    ("Maximum qualitative score", front["llm score"].idxmax()),
    ("Balanced compromise", distance_to_ideal.idxmin()),
]

## 3. Recalculation and validation

Each alternative is applied to `blocks_clean`. Its recalculated aggregate land-value gain must match the value stored in the Prompt 3 JSONL (within one ruble).

In [ ]:
base_blocks = blocks_clean.copy()
base_blocks["is_project"] = False
base_pred = LandPriceEstimator(blocks=base_blocks, **estimator_kwargs).predict()
base_cols = base_pred[["id", "land_value"]].rename(
    columns={"land_value": "land_value_before"}
)
baseline_total = float(base_pred["land_value"].sum())

def evaluate_scenario(scenario):
    target_id = int(scenario["target_id"])
    after_blocks = ScenarioTEPModifier(base_blocks).apply(
        target_id, dict(scenario["params_repaired"])
    )
    after_blocks["is_project"] = False
    after_blocks.loc[after_blocks["id"] == target_id, "is_project"] = True
    values = LandPriceEstimator(blocks=after_blocks, **estimator_kwargs).predict()
    values = values.merge(base_cols, on="id", how="left", validate="one_to_one")
    values["delta_rub"] = values["land_value"] - values["land_value_before"]
    values["delta_pct"] = np.where(
        values["land_value_before"] > 0,
        values["delta_rub"] / values["land_value_before"] * 100,
        np.nan,
    )
    values = values.replace([np.inf, -np.inf], np.nan)

    gain = float(values["delta_rub"].sum())
    error = gain - float(scenario["land_value_gain"])
    assert abs(error) < 1.0, (
        f"{scenario['scenario_id']} (seed {scenario['seed']}): "
        f"recalculated gain differs from JSONL by {error:,.2f} RUB"
    )

    target = values.loc[values["id"] == target_id].iloc[0]
    tolerance = 1e-6
    stats = {
        "selection_role": scenario.get("selection_role", ""),
        "scenario_id": scenario["scenario_id"],
        "algorithm_variant": scenario["algorithm_variant"],
        "seed": int(scenario["seed"]),
        "land_use": scenario["land_use"],
        "investor_npv": float(scenario["investor_npv"]),
        "land_value_gain": gain,
        "llm_score": float(scenario["llm score"]),
        "target_value_before": float(target["land_value_before"]),
        "target_value_after": float(target["land_value"]),
        "positive_blocks": int((values["delta_rub"] > tolerance).sum()),
        "negative_blocks": int((values["delta_rub"] < -tolerance).sum()),
        "validation_error_rub": error,
    }
    return values, stats

evaluated_front = {}
for index, scenario in front.iterrows():
    evaluated_front[index] = evaluate_scenario(scenario)

target_value_after = pd.Series({
    index: result[1]["target_value_after"]
    for index, result in evaluated_front.items()
})
selection = selection_rules + [("Maximum target-block value", target_value_after.idxmax())]
representatives = pd.DataFrame([front.loc[index] for _, index in selection]).reset_index(drop=True)
representatives.insert(0, "selection_role", [role for role, _ in selection])
assert len(representatives[["scenario_id", "seed"]].drop_duplicates()) == 5

scenario_results = [evaluated_front[index][0] for _, index in selection]
scenario_stats = pd.DataFrame([
    evaluated_front[index][1] | {"selection_role": role}
    for role, index in selection
])
assert (scenario_stats["validation_error_rub"].abs() < 1.0).all()

selection_table = scenario_stats[[
    "selection_role", "scenario_id", "algorithm_variant", "seed", "land_use",
    "investor_npv", "land_value_gain", "llm_score", "target_value_after",
]].copy()
selection_table["NPV, billion RUB"] = selection_table.pop("investor_npv") / 1e9
selection_table["Land-value gain, billion RUB"] = selection_table.pop("land_value_gain") / 1e9
selection_table["Target-block value, million RUB"] = selection_table.pop("target_value_after") / 1e6
selection_table = selection_table.rename(columns={
    "algorithm_variant": "Algorithm variant",
    "llm_score": "LLM score",
})
display(Markdown("### 1. Best NSGA-II alternatives with LLM"))
display(selection_table.style.format({
    "NPV, billion RUB": "{:.3f}",
    "Land-value gain, billion RUB": "{:.3f}",
    "LLM score": "{:.2f}",
    "Target-block value, million RUB": "{:.3f}",
}))

no_llm_evaluated = {
    index: evaluate_scenario(scenario)
    for index, scenario in no_llm_front.iterrows()
}
combined_stats = pd.concat([
    pd.DataFrame([result[1] for result in evaluated_front.values()]),
    pd.DataFrame([result[1] for result in no_llm_evaluated.values()]),
], ignore_index=True)

best_rows = []
comparison_objectives = ["investor_npv", "land_value_gain", "llm_score"]
for algorithm_variant, pool in combined_stats.groupby("algorithm_variant", sort=False):
    pool_normalized = (pool[comparison_objectives] - pool[comparison_objectives].min()) / (
        pool[comparison_objectives].max() - pool[comparison_objectives].min()
    )
    pool_distance = np.sqrt(((1 - pool_normalized) ** 2).sum(axis=1))
    rules = [
        ("Maximum NPV", pool["investor_npv"].idxmax()),
        ("Maximum land-value gain", pool["land_value_gain"].idxmax()),
        ("Maximum qualitative score", pool["llm_score"].idxmax()),
        ("Balanced compromise", pool_distance.idxmin()),
        ("Maximum target-block value", pool["target_value_after"].idxmax()),
    ]
    best_rows.extend(
        pool.loc[index].to_dict() | {"selection_role": role}
        for role, index in rules
    )

best_all_algorithms_table = pd.DataFrame(best_rows)[[
    "algorithm_variant", "selection_role", "scenario_id", "seed", "land_use",
    "investor_npv", "land_value_gain", "llm_score", "target_value_after",
]].copy()
best_all_algorithms_table["NPV, billion RUB"] = best_all_algorithms_table.pop("investor_npv") / 1e9
best_all_algorithms_table["Land-value gain, billion RUB"] = best_all_algorithms_table.pop("land_value_gain") / 1e9
best_all_algorithms_table["Target-block value, million RUB"] = best_all_algorithms_table.pop("target_value_after") / 1e6
best_all_algorithms_table = best_all_algorithms_table.rename(columns={
    "algorithm_variant": "Algorithm variant",
    "selection_role": "Selection role",
    "scenario_id": "Scenario ID",
    "seed": "Seed",
    "land_use": "Land use",
    "llm_score": "LLM score",
})
assert set(best_all_algorithms_table["Algorithm variant"]) == {
    "NSGA-II with LLM", "NSGA-II without LLM"
}
without_llm_table = best_all_algorithms_table.loc[
    best_all_algorithms_table["Algorithm variant"] == "NSGA-II without LLM"
].reset_index(drop=True)
assert len(without_llm_table) == 5
display(Markdown("### 2. Best NSGA-II alternatives without LLM"))
display(without_llm_table.style.format({
    "NPV, billion RUB": "{:.3f}",
    "Land-value gain, billion RUB": "{:.3f}",
    "LLM score": "{:.2f}",
    "Target-block value, million RUB": "{:.3f}",
}))

global_normalized = (
    combined_stats[comparison_objectives] - combined_stats[comparison_objectives].min()
) / (
    combined_stats[comparison_objectives].max() - combined_stats[comparison_objectives].min()
)
global_distance = np.sqrt(((1 - global_normalized) ** 2).sum(axis=1))
global_rules = [
    ("Maximum NPV", combined_stats["investor_npv"].idxmax()),
    ("Maximum land-value gain", combined_stats["land_value_gain"].idxmax()),
    ("Maximum qualitative score", combined_stats["llm_score"].idxmax()),
    ("Balanced compromise", global_distance.idxmin()),
    ("Maximum target-block value", combined_stats["target_value_after"].idxmax()),
]
global_best_table = pd.DataFrame([
    combined_stats.loc[index].to_dict() | {"selection_role": role}
    for role, index in global_rules
])[[
    "algorithm_variant", "selection_role", "scenario_id", "seed", "land_use",
    "investor_npv", "land_value_gain", "llm_score", "target_value_after",
]].copy()
assert len(global_best_table[["scenario_id", "seed"]].drop_duplicates()) == 5
global_best_table["NPV, billion RUB"] = global_best_table.pop("investor_npv") / 1e9
global_best_table["Land-value gain, billion RUB"] = global_best_table.pop("land_value_gain") / 1e9
global_best_table["Target-block value, million RUB"] = global_best_table.pop("target_value_after") / 1e6
global_best_table = global_best_table.rename(columns={
    "algorithm_variant": "Algorithm variant",
    "selection_role": "Selection role",
    "scenario_id": "Scenario ID",
    "seed": "Seed",
    "land_use": "Land use",
    "llm_score": "LLM score",
})
display(Markdown("### 3. Five best alternatives overall"))
display(global_best_table.style.format({
    "NPV, billion RUB": "{:.3f}",
    "Land-value gain, billion RUB": "{:.3f}",
    "LLM score": "{:.2f}",
    "Target-block value, million RUB": "{:.3f}",
}))

delta_metrics = [
    "NPV, billion RUB",
    "Land-value gain, billion RUB",
    "LLM score",
    "Target-block value, million RUB",
]
comparison_by_role = best_all_algorithms_table.pivot(
    index="Selection role", columns="Algorithm variant", values=delta_metrics
)
delta_table = pd.DataFrame({"Selection role": [role for role, _ in selection]})
delta_labels = {
    "NPV, billion RUB": "NPV delta, %",
    "Land-value gain, billion RUB": "Land-value gain delta, %",
    "LLM score": "LLM score delta, %",
    "Target-block value, million RUB": "Target-block value delta, %",
}
for metric, label in delta_labels.items():
    with_llm = comparison_by_role[(metric, "NSGA-II with LLM")]
    without_llm = comparison_by_role[(metric, "NSGA-II without LLM")]
    relative_delta = (with_llm - without_llm) / without_llm * 100
    delta_table[label] = delta_table["Selection role"].map(relative_delta)

assert len(delta_table) == 5 and delta_table.notna().all().all()
display(Markdown(
    "### 4. Algorithm deltas, % ((with LLM − without LLM) / without LLM × 100)"
))
display(delta_table.style.format({
    column: "{:+.2f}%" for column in delta_table.columns if column.endswith("delta, %")
}))

## 4. Target-block context map

The context extent is reused for all five spatial-effect maps.

In [ ]:
target_id = int(representatives["target_id"].iloc[0])
target_block = blocks_clean.loc[blocks_clean["id"] == target_id]
metric_crs = blocks_clean.estimate_utm_crs() if blocks_clean.crs.is_geographic else blocks_clean.crs
context_geometry = target_block.to_crs(metric_crs).buffer(4000).to_crs(blocks_clean.crs).iloc[0]
context_ids = blocks_clean.loc[blocks_clean.geometry.intersects(context_geometry), "id"]
context = blocks_clean.loc[blocks_clean["id"].isin(context_ids)].copy()
context_target = context.loc[context["id"] == target_id]
target_centroid = context_target.to_crs(metric_crs).centroid.to_crs(context.crs)
extent = context.total_bounds

fig, ax = plt.subplots(figsize=(9, 9))
context.plot(
    ax=ax, column="land_value_per_100m2", cmap="viridis", legend=True,
    legend_kwds={"label": "Baseline land value, RUB per 100 m²", "shrink": 0.72},
    edgecolor="white", linewidth=0.35,
)
context_target.boundary.plot(ax=ax, color="red", linewidth=2.5, zorder=3)
target_centroid.plot(ax=ax, color="white", edgecolor="red", markersize=50, zorder=4)
ax.set(xlim=(extent[0], extent[2]), ylim=(extent[1], extent[3]))
ax.set_title(f"Target block {target_id} and local context", fontsize=15)
ax.set_axis_off()
fig.tight_layout()
plt.show()

## 5. Sequential spatial-effect maps

The four representative alternatives are displayed one after another using the same map style and a fixed symmetric percentage-change scale.

In [ ]:
legend_labels = {
    "Кварталы без существующих изменений": "Blocks without land-value change",
    "Кварталы с изменением цены": "Blocks with land-value change",
    "Границы изменяемого квартала": "Target block boundary",
}

for (_, scenario), values, stats in zip(
    representatives.iterrows(), scenario_results, scenario_stats.to_dict("records")
):
    print(f"{scenario['selection_role']} — {scenario['land_use'].title()} (seed {int(scenario['seed'])})")
    print(
        f"NPV: {stats['investor_npv'] / 1e9:.3f} billion RUB | "
        f"Land-value gain: {stats['land_value_gain'] / 1e9:.3f} billion RUB | "
        f"LLM score: {stats['llm_score']:.2f}"
    )
    print(
        f"Target block value: {stats['target_value_before'] / 1e6:.3f} → "
        f"{stats['target_value_after'] / 1e6:.3f} million RUB | "
        f"Positive/negative blocks: {stats['positive_blocks']}/{stats['negative_blocks']}"
    )

    result = plot_scenario_impact(
        blocks=values,
        target_idx=target_id,
        target_id_column="id",
        pct_column="delta_pct",
        delta_column="delta_rub",
        print_summary=False,
        print_quarter_stats=False,
        figsize=(22, 17),
        show=False,
    )
    fig = result["fig"]
    ax = fig.axes[0]
    ax.set_title(
        f"{scenario['selection_role']}: {scenario['land_use'].title()} "
        f"(seed {int(scenario['seed'])})\nLand-value change, %",
        pad=12,
        fontsize=24,
    )

    if len(fig.axes) > 1:
        fig.axes[-1].set_ylabel("Land-value change, %", fontsize=18)

    legend = ax.get_legend()
    if legend is not None:
        legend.set_title("Legend", prop={"size": 16})
        for text in legend.get_texts():
            text.set_text(legend_labels.get(text.get_text(), text.get_text()))

    for text in list(ax.texts):
        if text.get_text() == "Масштаб недоступен (географическая проекция)":
            text.set_text("")
        elif text.get_text() == "":
            text.set_text("Scale")
        elif text.get_text().endswith(" м"):
            text.set_text(text.get_text()[:-2] + " m")
        else:
            text.remove()

    display(fig)
    plt.close(fig)

## 6. Economically matched urban-planning comparison

This comparison uses the complete Prompt 3 Pareto fronts for seeds 32–41 rather than only the aggregated global front. Within each seed, an LLM and a no-LLM alternative are considered economically comparable when their Euclidean distance in the jointly normalized NPV–land-value-gain space is at most `0.10`. The highest-scoring eligible LLM alternative is compared with the highest-scoring compatible no-LLM alternative, making the qualitative comparison conservative. Seeds without a pair inside the fixed caliper are reported as unmatched rather than forced into the comparison.

The LLM score measures the Prompt 3 active-urban-life objective. Functional-mix entropy, the combined business-and-recreation share, and MXI are shown as additional planning-oriented descriptors; they should not be interpreted as direct measures of pedestrian accessibility or public-space quality.

In [ ]:
comparison_seeds = range(32, 42)
economic_cols = ["investor_npv", "land_value_gain"]
land_use_share_cols = [
    "residential", "business", "recreation", "industrial",
    "transport", "special", "agriculture",
]
economic_caliper = 0.10

def load_prompt_3_fronts(seed):
    llm_seed = pd.read_json(
        ROOT / f"examples/nsga_2_llm_prompts/seed_{seed}/prompt_3/prompt_3_pareto_front.jsonl",
        lines=True,
    )
    llm_seed["seed"] = seed
    llm_seed["scenario_id"] = [
        f"prompt_3_seed_{seed}_{index}" for index in range(len(llm_seed))
    ]
    no_llm_seed = pd.read_json(
        ROOT / (
            f"examples/nsga_2_without_llm/seed_{seed}/prompt_3/"
            f"no_llm_pareto_front_{seed}_scored_by_prompt_3.jsonl"
        ),
        lines=True,
    )
    return llm_seed, no_llm_seed

llm_seed_fronts, no_llm_seed_fronts = zip(*(
    load_prompt_3_fronts(seed) for seed in comparison_seeds
))
llm_complete = pd.concat(llm_seed_fronts, ignore_index=True)
no_llm_complete = pd.concat(no_llm_seed_fronts, ignore_index=True)
economic_pool = pd.concat(
    [llm_complete[economic_cols], no_llm_complete[economic_cols]],
    ignore_index=True,
)
economic_min = economic_pool.min()
economic_range = (economic_pool.max() - economic_min).replace(0, 1)

def planning_descriptors(scenario):
    params = scenario["params_repaired"]
    shares = np.array([max(float(params.get(name, 0)), 0) for name in land_use_share_cols])
    shares = shares / shares.sum()
    nonzero = shares[shares > 0]
    entropy = float(-(nonzero * np.log(nonzero)).sum() / np.log(len(shares)))
    return {
        "functional_mix_entropy": entropy,
        "active_use_share_pct": 100 * (float(params["business"]) + float(params["recreation"])),
        "mxi": float(params["mxi"]),
    }

matched_rows = []
unmatched_seeds = []
for seed in comparison_seeds:
    llm_seed = llm_complete.loc[llm_complete["seed"] == seed]
    no_llm_seed = no_llm_complete.loc[no_llm_complete["seed"] == seed]
    eligible_pairs = []
    for llm_index, llm_scenario in llm_seed.iterrows():
        for no_llm_index, no_llm_scenario in no_llm_seed.iterrows():
            distance = float(np.linalg.norm(
                ((llm_scenario[economic_cols] - no_llm_scenario[economic_cols]) / economic_range)
                .to_numpy(dtype=float)
            ))
            if distance <= economic_caliper:
                eligible_pairs.append((llm_index, no_llm_index, distance))

    if not eligible_pairs:
        unmatched_seeds.append(seed)
        continue

    eligible_llm_indices = {pair[0] for pair in eligible_pairs}
    best_llm_index = max(
        eligible_llm_indices, key=lambda index: llm_complete.loc[index, "llm score"]
    )
    compatible_pairs = [pair for pair in eligible_pairs if pair[0] == best_llm_index]
    _, best_no_llm_index, economic_distance = max(
        compatible_pairs,
        key=lambda pair: (
            no_llm_complete.loc[pair[1], "llm score"], -pair[2]
        ),
    )
    llm_scenario = llm_complete.loc[best_llm_index]
    no_llm_scenario = no_llm_complete.loc[best_no_llm_index]
    llm_planning = planning_descriptors(llm_scenario)
    no_llm_planning = planning_descriptors(no_llm_scenario)

    matched_rows.append({
        "Seed": seed,
        "LLM scenario": llm_scenario["scenario_id"],
        "No-LLM scenario": no_llm_scenario["scenario_id"],
        "Economic distance": economic_distance,
        "NPV change, %": (llm_scenario["investor_npv"] / no_llm_scenario["investor_npv"] - 1) * 100,
        "Land-value-gain change, %": (llm_scenario["land_value_gain"] / no_llm_scenario["land_value_gain"] - 1) * 100,
        "LLM score: with LLM": llm_scenario["llm score"],
        "LLM score: without LLM": no_llm_scenario["llm score"],
        "Qualitative-score change, pp": llm_scenario["llm score"] - no_llm_scenario["llm score"],
        "Functional-mix entropy change": llm_planning["functional_mix_entropy"] - no_llm_planning["functional_mix_entropy"],
        "Business + recreation change, pp": llm_planning["active_use_share_pct"] - no_llm_planning["active_use_share_pct"],
        "MXI change": llm_planning["mxi"] - no_llm_planning["mxi"],
    })

matched_comparison = pd.DataFrame(matched_rows)
assert (matched_comparison["Economic distance"] <= economic_caliper + 1e-12).all()
assert matched_comparison["Seed"].is_unique

display(Markdown(
    f"**Matched seeds:** {len(matched_comparison)}/{len(list(comparison_seeds))}; "
    f"**unmatched seeds:** {', '.join(map(str, unmatched_seeds))}."
))
display(matched_comparison.style.format({
    "Economic distance": "{:.3f}",
    "NPV change, %": "{:+.2f}%",
    "Land-value-gain change, %": "{:+.2f}%",
    "LLM score: with LLM": "{:.2f}",
    "LLM score: without LLM": "{:.2f}",
    "Qualitative-score change, pp": "{:+.2f}",
    "Functional-mix entropy change": "{:+.3f}",
    "Business + recreation change, pp": "{:+.2f}",
    "MXI change": "{:+.3f}",
}))

summary_rows = []
for metric in [
    "Qualitative-score change, pp",
    "Functional-mix entropy change",
    "Business + recreation change, pp",
    "MXI change",
    "NPV change, %",
    "Land-value-gain change, %",
]:
    values = matched_comparison[metric]
    summary_rows.append({
        "Metric": metric,
        "Median change": values.median(),
        "LLM better, seeds": f"{int((values > 0).sum())}/{len(values)}",
    })
comparison_summary = pd.DataFrame(summary_rows)
display(Markdown("### Across-seed summary"))
display(comparison_summary.style.format({"Median change": "{:+.3f}"}))

plot_metrics = [
    "Qualitative-score change, pp",
    "Business + recreation change, pp",
    "NPV change, %",
    "Land-value-gain change, %",
]
fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=False)
for ax, metric in zip(axes.flat, plot_metrics):
    values = matched_comparison.set_index("Seed")[metric]
    colors = ["#2f5d8a" if value >= 0 else "#c8d6e5" for value in values]
    ax.bar(values.index.astype(str), values, color=colors, edgecolor="#263238", linewidth=0.6)
    ax.axhline(0, color="#263238", linewidth=0.8)
    ax.set_title(metric, fontsize=11)
    ax.set_xlabel("Seed")
    ax.grid(axis="y", color="#e5e7eb", linewidth=0.7)
fig.suptitle(
    "NSGA-II with LLM minus economically matched NSGA-II without LLM",
    fontsize=15,
)
fig.tight_layout()
plt.show()

qualitative = matched_comparison["Qualitative-score change, pp"]
active_use = matched_comparison["Business + recreation change, pp"]
mxi_delta = matched_comparison["MXI change"]
npv_delta = matched_comparison["NPV change, %"]
land_value_delta = matched_comparison["Land-value-gain change, %"]
entropy_delta = matched_comparison["Functional-mix entropy change"]
display(Markdown(
    "### Interpretation\n\n"
    f"Within the strict matched subset, the LLM-guided alternatives have a median "
    f"qualitative-score change of **{qualitative.median():+.2f} points** and a higher "
    f"score in **{int((qualitative > 0).sum())}/{len(qualitative)} seeds**. "
    f"Business and recreation share increases in **{int((active_use > 0).sum())}/{len(active_use)} seeds** "
    f"(median **{active_use.median():+.2f} percentage points**), and MXI increases in "
    f"**{int((mxi_delta > 0).sum())}/{len(mxi_delta)} seeds** "
    f"(median **{mxi_delta.median():+.3f}**). The median economic trade-off is "
    f"**{npv_delta.median():+.2f}%** for NPV and **{land_value_delta.median():+.2f}%** "
    f"for land-value gain. Functional-mix entropy improves in only "
    f"**{int((entropy_delta > 0).sum())}/{len(entropy_delta)} seeds**, so these data do not support "
    "a general claim of higher land-use entropy. This is an illustrative matched comparison; "
    "the multi-seed three-objective hypervolume remains the primary aggregate result."
))